# Notebook 5: One Class Network

## Pre-requisitos

### Instalar paquetes

Si la práctica requiere algún paquete de Python, habrá que incluir una celda en la que se instalen. Si usamos un paquete que se ha utilizado en prácticas anteriores, podríamos dar por supuesto que está instalado pero no cuesta nada satisfacer todas las dependencias en la propia práctica para reducir las dependencias entre ellas.

In [2]:
# Ejemplo de instalación de tensorflow 2.0
#%tensorflow_version 2.x
# !pip3 install tensorflow  # NECESARIO SOLO SI SE EJECUTA EN LOCAL
import tensorflow as tf

# Hacemos los imports que sean necesarios
import numpy as np

# One Class sobre datos artificiales

Lo primero que tenemos que hacer es definir los datos a utilizar.

In [3]:
random_state = 42
rng = np.random.RandomState(random_state)
#  datos de entrenamiento
X = 0.3 * rng.randn(5000, 2)
x_train = np.r_[X + 2, X - 2]
#  datos de test en la misma distribución que los datos de entrenamiento
X = 0.3 * rng.randn(200, 2)
x_test = np.r_[X + 2, X - 2]
#  outliers
x_outliers = rng.uniform(low=-4, high=4, size=(200, 2))

## Crea tu propia red para la detección de anomalías

Vamos a crear nuestra propia red para la detección de anomalías. Para ello, vamos a definir una red cualquiera, que nos **transforme los datos de entrada en una salida de un único elemento**. Esta red va a cumplir una serie de características:

* La capa anterior a la salida serán las llamadas **deep features**.
* Todas las capas (incluyendo la última) deben incluir regularización.
* La función de coste es $$L(y, \tilde{y}) = \dfrac{1}{2} \| w^2 \| + \dfrac{1}{\nu} \dfrac{1}{N} \sum_{i=1}^N \max(0, r - \tilde{y}) $$ donde $\tilde{y}$ es la salida de la red, $\nu$ es un hiperparámetro entre 0 y 1, y $r$ es un parámetro no entrenable, pero que va a ser modificado en cada epoch.
* Al final del cada epoch, r va a ser modificado al valor del $\nu$-cuantil de los datos de entrada (este valor será modificado gracias al Callback proporcionado a continuación).
* Para la predicción, se considerará un dato típico si $\tilde{y} > r$. En caso contrario, será un dato atípico.

In [4]:
class ChangeRCallback(tf.keras.callbacks.Callback):
   def __init__(self, train_data, delta=.025, steps=3):
       super().__init__()
       self.train_data = train_data
       self.delta = delta
       self.steps = steps
       self.cont = 0

   def on_epoch_end(self, epoch, logs=None):
       sorted_values = np.sort(self.model.predict(self.train_data).flatten())
       new_value = sorted_values[int(len(sorted_values) * (1. - self.model.nu))]
       old_value = self.model.r.numpy()
       print('Cambiando r a', new_value, ', max:', sorted_values.max(), ', min:', sorted_values.min())
       self.model.r.assign(new_value)
       if np.abs(old_value - new_value) < self.delta:
            self.cont += 1
            if self.cont >= self.steps:
                print('Convergencia obtenida. Finalizando el entrenamiento.')
                self.model.stop_training = True
       else:
            self.cont = 0

Tu trabajo es crear el modelo y entrenarlo.

In [5]:

class DetectorAnomalias:

    def __init__(self, input_shape, nu=.5):
        l2_val = 0.0001
        input_shape = (input_shape, )
        inputs = tf.keras.layers.Input(shape=input_shape)
        x = tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(inputs)
        x = tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(x)
        deep_features = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(x)
        outputs = tf.keras.layers.Dense(1, activation='sigmoid', kernel_regularizer=tf.keras.regularizers.L2(l2_val))(deep_features)

        self.encoder = tf.keras.models.Model(inputs, deep_features)
        self.model = tf.keras.models.Model(inputs, outputs)

        self.model.r = tf.Variable(1.0, trainable=False, name='r', dtype=tf.float32)
        self.model.nu = tf.Variable(nu, trainable=False, name='nu', dtype=tf.float32)

        self.model.compile(optimizer="Adam",
                           loss=self.loss_function)

    def loss_function(self, y_true, y_pred):
        l2 = 0.0
        for var in self.model.trainable_weights:
            l2 += tf.reduce_sum(tf.square(var))

        reg_loss = 0.5 * l2
        return reg_loss + (1 / self.model.nu * tf.cast(tf.shape(y_true)[0], tf.float32)) * tf.reduce_sum(tf.maximum(0.0, self.model.r - y_pred))
    
    def fit(self, X, y=None, sample_weight=None):
        dummy_y = np.zeros((len(X), 1))

        return self.model.fit(
            X,
            dummy_y,
            epochs=20,
            batch_size=32,
            verbose=1,
            callbacks=[
                ChangeRCallback(X)
            ]
        )
        
    def predict(self, X):
        return self.model(X)
    
    def get_encoded_data(self, X):
        return self.encoder(X)
        
    def __del__(self):
        tf.keras.backend.clear_session()

### Entrena el modelo.

Usa lo hecho anteriormente para entrenar tu modelo.

In [10]:
Modelo = DetectorAnomalias(2, 0.9)

In [11]:
Modelo.fit(x_train)

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 465us/step - loss: 139
Cambiando r a 0.999518 , max: 0.9999956 , min: 0.995007
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 54.4451   
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 378us/step - loss: 17
Cambiando r a 0.9991084 , max: 0.99998355 , min: 0.9929139
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 16.0924  
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 388us/step - loss: 12
Cambiando r a 0.9987578 , max: 0.999967 , min: 0.99159926
Convergencia obtenida. Finalizando el entrenamiento.
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 12.1235  


## Evaluando el modelo

Una vez entrenado, para evaluar el modelo sólo hay que tener en cuenta lo siguiente:

  1. Si la salida es mayor que r, es un dato típico.
  1. Si la salida es menor que r, es un dato atípico.

### TRABAJO: Evalúa el modelo con los datos del conjunto de test, y con los outliers. Visualiza los datos típicos y atípicos con una gráfica.

In [8]:
print(np.count_nonzero(Modelo.predict(x_outliers) > Modelo.model.r))
print(np.count_nonzero(Modelo.predict(x_test) > Modelo.model.r))

76
361


In [9]:
# TODO: Evalúa el modelo con los datos del conjunto de outliers. Indica el porcentaje de datos etiquetados como atípicos, y visualiza los datos en conjunto con los de test


¿Qué resultados has obtenido? Si el número de outliers detectado es bajo (inferior al 30%), puedes estar cometiendo algún error, entre ellos:

* Sobreentrenar el modelo. Prueba a usar un delta distinto en el callback.
* Usar un valor de $\nu$ demasiado alto.

Prueba distintas configuraciones para ver su efecto.